# P.R.I.S.M. - Backend Evaluation Notebook

This notebook is designed for judges and evaluators to test the P.R.I.S.M. (Probabilistic Reasoning and Interpretability System for Models) backend API. The frontend is deployed on Vercel, so this notebook only sets up and exposes the backend.

### Instructions:
1. Ensure your Kaggle notebook has the **T4 x2** accelerator enabled in the Session Options.
2. Ensure **Internet** is toggled **On**.
3. Run all the cells below in order.
4. The final cell will generate a public URL for the backend.
5. **Copy the backend URL** and provide it to the Vercel frontend at: [Your Vercel Frontend URL]
6. *Note: If LocalTunnel prompts you for an "Endpoint IP", copy and paste the IP address printed right above the link.*

In [1]:
# 1. Setup Environment & Clone Repository
!echo "Installing Node.js..."
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
!apt-get install -y nodejs

!echo "\nCloning P.R.I.S.M. repository..."
!git clone https://github.com/chandan989/P.R.I.S.M..git

Installing Node.js...
2026-05-12 18:34:29 - Installing pre-requisites
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 https://cli.github.com/packages stable InRelease [3,917 B]               
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]      
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,251 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/multiverse amd64 Packages [86.0 kB]
Get:12 http://archiv

In [2]:
# 2. Install llama-cpp-python and Download Model
!echo "Installing llama-cpp-python with CUDA 12.1 support..."
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 huggingface_hub

Installing llama-cpp-python with CUDA 12.1 support...
Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 GB 920.0 kB/s eta 0:00:0000:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00


In [3]:
!echo "\nDownloading P.R.I.S.M. MXFP4 GGUF model..."
# !mkdir -p models

!pip install -U "huggingface_hub[cli]"

import os
from huggingface_hub import snapshot_download

print("\nDownloading P.R.I.S.M. MXFP4 GGUF model...")
os.makedirs("models", exist_ok=True)

snapshot_download(
    repo_id="chandan989/prism-gemma-4-26B-A4B-it-MXFP4-v3.5",
    local_dir="models",
    token=""
)
# snapshot_download(repo_id=repo_id, local_dir="/content/model")

\nDownloading P.R.I.S.M. MXFP4 GGUF model...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 96.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 40.0 MB/s eta 0:00:00
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1



Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

'/kaggle/working/models'

In [4]:
# 3. Configure and Start the FastAPI Backend
import subprocess
import time
import os

# Configure .env to use llama_cpp instead of ollama
env_path = "P.R.I.S.M./backend/.env"
!cp P.R.I.S.M./backend/.env.example {env_path}
!sed -i 's|MODEL_BACKEND=ollama|MODEL_BACKEND=llama_cpp|g' {env_path}
!sed -i 's|MODEL_PATH=|MODEL_PATH=../../models/|g' {env_path}
!sed -i 's|KB_ROOT=./knowledge_base|KB_ROOT=../knowledge_base|g' {env_path}

print("Installing Python backend dependencies...")
!pip install -r P.R.I.S.M./backend/requirements.txt

print("\nStarting FastAPI backend server on port 8000 (Model will load into VRAM now, please wait)...\n")
backend_process = subprocess.Popen(
    ["python3", "server.py", "--port", "8000"],
    cwd="P.R.I.S.M./backend"
)
time.sleep(15) # Wait for backend to initialize and model to load into VRAM

Installing Python backend dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 61.9 MB/s eta 0:00:00:00:0100:01

Starting FastAPI backend server on port 8000 (Model will load into VRAM now, please wait)...



INFO:     Will watch for changes in these directories: ['/kaggle/working/P.R.I.S.M./backend']
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [1818] using WatchFiles
INFO:     Started server process [1823]
INFO:     Waiting for application startup.
INFO:server:Initializing P.R.I.S.M. components...
INFO:server:Initializing Gemma client with backend: llama_cpp
INFO:watchfiles.main:3 changes detected
INFO:client.gemma_client:Loading llama.cpp model from: ../../models/prism-gemma-4-26B-A4B-it-MXFP4-v3.5.gguf


In [ ]:
# 4. Verify Backend Health and Real Audit Streaming
import json
import time
import urllib.request

print("Waiting for /health to become ready...")
health = None
for attempt in range(24):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=10) as res:
            health = json.loads(res.read().decode("utf-8"))
        break
    except Exception as exc:
        print(f"Attempt {attempt + 1}/24: backend not ready yet ({exc})")
        time.sleep(5)

if health is None:
    raise RuntimeError("Backend did not become healthy on http://127.0.0.1:8000/health")

print(json.dumps(health, indent=2)[:4000])

payload = json.dumps({
    "query": "Patient taking warfarin and clarithromycin. Identify the key interaction.",
    "max_tokens": 256,
}).encode("utf-8")

req = urllib.request.Request(
    "http://127.0.0.1:8000/api/audit",
    data=payload,
    headers={"Content-Type": "application/json"},
    method="POST",
)

print("\nTesting /api/audit SSE stream. This may take a minute while the model generates...")
seen = []
with urllib.request.urlopen(req, timeout=240) as res:
    for raw in res:
        line = raw.decode("utf-8").strip()
        if not line.startswith("data:"):
            continue
        event = json.loads(line[5:].strip())
        event_type = event.get("type")
        seen.append(event_type)
        print("SSE event:", event_type)
        if event_type in {"error", "done"}:
            if event_type == "error":
                raise RuntimeError(event.get("content", "Unknown /api/audit error"))
            break

required = {"thought", "answer", "confidence", "done"}
missing = required.difference(seen)
if missing:
    raise RuntimeError(f"/api/audit stream missing expected event types: {sorted(missing)}")

print("Backend smoke test passed.")


In [ ]:
# 5. Expose Backend via LocalTunnel
import urllib.request

print("\n" + "="*60)
print("🚀 BACKEND READY: GENERATING PUBLIC URL")
print("="*60)

ipv4 = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n")
print(f"\n\033[1mIMPORTANT:\033[0m When you click the link below, you may be asked to enter a Tunnel Password.")
print(f"Your Tunnel Password / Endpoint IP is: \033[1;32m{ipv4}\033[0m\n")

print("\033[1mNEXT STEPS:\033[0m")
print("1. Copy the backend URL from the link below")
print("2. Go to your Vercel frontend")
print("3. Enter the backend URL in the configuration/settings")
print("4. Start testing the P.R.I.S.M. system!\n")

# Expose the backend port (8000)
!npx --yes localtunnel --port 8000

# Frontend Link: https://p-r-i-s-m-pearl.vercel.app/